In [22]:
from collections import defaultdict

import cv2 as cv

from ultralytics import YOLO

#Load the YOLO model
model=YOLO("yolo26n.pt")


In [23]:
class_list=model.names
class_list

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [24]:
#Open the video
cap=cv.VideoCapture('videos/test_video_2.mp4')
track_history = defaultdict(lambda: [])
counting_line_y=450

#Dictionary to store objects counts by classes 
class_counts=defaultdict(int)

#Dictionary to keep track of objects that crossed the line 
crossed_ids=set()

while cap.isOpened():
    ret,frame=cap.read()
    if not ret:
        break
    frame = cv.resize(frame, (640, 640))
    results=model.track(frame,persist=True,classes=[1,2,3,5,6,7])[0]
    # print(results)
    if results.boxes and results.boxes.is_track:
            boxes = results.boxes.xywh.cpu()
            track_ids = results.boxes.id.int().cpu().tolist()
            class_ids=results.boxes.cls.cpu().tolist()
        
            # Visualize the result on the frame
            frame = results.plot()
            # Draw a counting line
            cv.line(frame,(5,counting_line_y),(600,counting_line_y),(0,0,255),5)
            # Loop through each detected object
            for box, track_id, class_id in zip(boxes, track_ids,class_ids):
                x, y, w, h = box
                track = track_history[track_id]
                # x, y center point
                track.append((float(x), float(y)))  
                # Retain 30 tracks for 30 frames
                if len(track) > 30: 
                    track.pop(0)
                cv.circle(frame,(int(x),int(y)),4,(0,0,255),-1)
                class_name=class_list[class_id]
                #Check if  the object has crossed the line
                if y>counting_line_y and track_id not in crossed_ids:
                    #Mark the object as crossed
                    crossed_ids.add(track_id)
                    #Increase the counter
                    class_counts[class_name]+=1
                
                #Display the counts on the frame
                y_offset=100  
                for class_name,count in class_counts.items():    
                    cv.putText(frame,
                    (f"{class_name}:{count}"),
                    (30,y_offset),
                    cv.FONT_HERSHEY_DUPLEX,
                    0.7,
                    (0,255,0),
                    2)
                    y_offset+=30
    #Display the frame
    cv.imshow("YOLO object detecting and tracking",frame)
    #Exit loop if "q" key is pressed
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

#Release resources
cap.release()
cv.destroyAllWindows()


0: 640x640 8 cars, 2 motorcycles, 1 truck, 55.1ms
Speed: 2.2ms preprocess, 55.1ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 7 cars, 2 motorcycles, 1 truck, 51.2ms
Speed: 2.3ms preprocess, 51.2ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 7 cars, 1 motorcycle, 96.7ms
Speed: 1.8ms preprocess, 96.7ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 8 cars, 1 motorcycle, 67.5ms
Speed: 3.0ms preprocess, 67.5ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 7 cars, 1 motorcycle, 1 truck, 64.5ms
Speed: 3.3ms preprocess, 64.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 7 cars, 1 motorcycle, 64.9ms
Speed: 3.0ms preprocess, 64.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 8 cars, 1 truck, 70.6ms
Speed: 2.6ms preprocess, 70.6ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x64